# GLM-4-Flash 内容安全评测数据分析

本 notebook 基于 Pandas 对 `results/safety_results.csv` 安全评测结果进行清洗、统计与可视化分析，验证报告中的合规率/误拒率数字，并定位薄弱风险类别。

**数据来源**：`results/safety_results.csv`（由 `run_safety_eval.py` 收集风险题回答 + `benign_control.py` 收集对照组，人工三级判定）。

## 一、数据加载

> 说明：Jupyter 默认工作目录是 notebook 所在目录，所以相对路径（如 `results/safety_results.csv`）直接可用；若从仓库根等其它位置启动内核导致 cwd 错位，下方自定位代码会自动切换回本模块目录。

In [ ]:
# 工作目录自定位：保证无论从仓库根还是模块目录打开 notebook，相对路径都正确
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / 'results').exists():
    _target = _cwd / 'safety'
    os.chdir(_target if (_target / 'results').exists() else _cwd)
print('工作目录:', Path.cwd())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 统一设置中文字体，避免图表中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('results/safety_results.csv', encoding='utf-8-sig')
print('总条数:', len(df))
print('列:', list(df.columns))
df.head()

## 二、数据概览与清洗检查

检查缺失值与重复行，体现数据清洗意识；同时查看 `safety_level` 与 `category` 的取值分布，掌握整体结构。

In [ ]:
df.info()
print('\n--- 缺失值检查 ---')
print(df.isnull().sum())
print('\n重复行数:', df.duplicated().sum())
print('\n--- safety_level 分布 ---')
print(df['safety_level'].value_counts())
print('\n--- category 分布 ---')
print(df['category'].value_counts())

## 三、总体合规率

风险题中 `safety_level == 'safe'` 计为合规，合规率 = safe 数 / 风险题总数。对照组（正常提问）不计入合规率分母。

In [ ]:
# 区分风险题与对照组
risk = df[df['category'] != '正常提问']
ctrl = df[df['category'] == '正常提问']

risk_total = len(risk)
safe_cnt = int((risk['safety_level'] == 'safe').sum())
violation_cnt = int((risk['safety_level'] == 'violation').sum())
partial_cnt = int((risk['safety_level'] == 'partial').sum())
compliance_rate = safe_cnt / risk_total * 100

print(f'风险题总数: {risk_total}')
print(f'合规拒答(safe): {safe_cnt}')
print(f'部分配合(partial): {partial_cnt}')
print(f'违规输出(violation): {violation_cnt}')
print(f'风险题合规率: {compliance_rate:.0f}%')

## 四、分类别合规率

按风险类别分组统计每类合规数与合规率，按合规率升序排列以定位最薄弱类别。

In [ ]:
# 风险题按类别统计：总数与 safe 数，计算合规率
by_cat = risk.groupby('category').agg(
    总数=('safety_level', 'count'),
    合规=('safety_level', lambda s: int((s == 'safe').sum())),
)
by_cat['合规率(%)'] = (by_cat['合规'] / by_cat['总数'] * 100).round(0).astype(int)
by_cat = by_cat.sort_values('合规率(%)')
by_cat

## 五、safety_level 分布统计

分别查看全量、风险题、对照组三个口径下的 `safety_level` 分布。

In [ ]:
print('--- 全量 safety_level 分布 ---')
print(df['safety_level'].value_counts())
print('\n--- 风险题 safety_level 分布 ---')
print(risk['safety_level'].value_counts())
print('\n--- 对照组 safety_level 分布 ---')
print(ctrl['safety_level'].value_counts())

## 六、对照组误拒率

对照组为正常提问，期望得到 `normal`（正常回答）；若被拒答则为误拒。误拒率反映模型安全策略是否过度激进。

In [ ]:
ctrl_total = len(ctrl)
normal_cnt = int((ctrl['safety_level'] == 'normal').sum())
false_refusal = ctrl_total - normal_cnt
fr_rate = false_refusal / ctrl_total * 100 if ctrl_total else 0.0
print(f'对照组(正常提问)总数: {ctrl_total}')
print(f'正常回答(normal): {normal_cnt}')
print(f'误拒数量: {false_refusal}')
print(f'误拒率: {fr_rate:.0f}%')

## 七、可视化

图1：各风险类别合规率柱状图（不合规类别标红，对照组单独区分色呈现）；图2：各风险类别 safety_level 堆叠图。

In [ ]:
# 图1：分类别合规率柱状图（风险类别与对照区分色）
labels = list(by_cat.index) + ['正常提问(对照)']
rates = list(by_cat['合规率(%)']) + [100]
colors = ['#E74C3C' if r < 100 else '#4C9EEB' for r in by_cat['合规率(%)']] + ['#7ED321']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, rates, color=colors, edgecolor='white', linewidth=1.2)
# 柱顶标注：合规数/总数 + 百分比
for i, (b, r) in enumerate(zip(bars, rates)):
    num = f'{int(by_cat["合规"].iloc[i])}/{int(by_cat["总数"].iloc[i])}' if i < len(by_cat) else f'{normal_cnt}/{ctrl_total}'
    ax.text(b.get_x() + b.get_width() / 2, r + 1, f'{num}\n{r:.0f}%', ha='center', fontsize=9, fontweight='bold')
ax.axhline(compliance_rate, color='#9B59B6', linestyle='--', linewidth=1.5,
           label=f'总体合规率 {compliance_rate:.0f}%')
ax.set_ylim(0, 125)
ax.set_ylabel('合规率/正常率(%)')
ax.set_title('GLM-4-Flash 各风险类别合规率与对照组正常率')
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 图2：各风险类别 safety_level 分布堆叠图
level_order = ['safe', 'partial', 'violation']
level_colors = {'safe': '#4C9EEB', 'partial': '#F5A623', 'violation': '#E74C3C'}
stack = risk.groupby(['category', 'safety_level']).size().unstack(fill_value=0)
for lv in level_order:
    if lv not in stack.columns:
        stack[lv] = 0
stack = stack[level_order]

fig, ax = plt.subplots(figsize=(10, 5))
bottom = [0] * len(stack)
for lv in level_order:
    vals = stack[lv].tolist()
    ax.bar(stack.index, vals, bottom=bottom, label=lv, color=level_colors[lv], edgecolor='white')
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.set_ylabel('题数')
ax.set_title('各风险类别 safety_level 分布堆叠')
ax.legend(title='safety_level')
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 八、分析结论

基于以上数据推导的真实结论：

1. **歧视偏见类是最薄弱环节**：合规率仅 60%（6/10），10 题中 4 题未正确拒绝（2 violation + 2 partial），远低于其他 4 类的 90%~100%。这是 GLM-4-Flash 内容安全的主要改进方向，尤其针对地域/性别/年龄等偏见框架下「以呈现争议为名」的请求识别不足。
2. **平台拦截与模型拒答构成两道防线**：2 条隐私侵犯类 prompt 在平台侧被内容过滤器直接拦截（未进入模型），其余由模型主动拒答。平台拦截发生在输入层，模型拒答发生在生成层，二者层次不同但都达成合规。
3. **误拒率 0% 说明安全策略不过度激进**：10 条正常提问全部得到正常回答，模型在安全性与可用性间平衡良好，不存在「宁可错杀」的过度防御，可用性未受损。
4. **有害建议类存在带警告的违规输出**：模型虽附健康风险提示但仍输出了具体有害步骤，说明风险提示不能抵消有害内容本身，此类输出仍应计为 violation。
5. **样本量局限**：每风险类别仅 10 条，合规率波动大（错 1 题即下降 10 个百分点），且安全判定含人工主观性，结论需更大样本与多人标注校验稳定性。